The project: building an out-of-core analytics pipeline (processing a dataset bigger than RAM) that can handle the whole of NYC's public yellow-taxi trip data from 2009 to 2025, which is ~50GB of data (much more than Colab's ~12.7GB RAM).

**Step 1:** installing DuckDB to query the Parquet datasource.

In [2]:
!pip install duckdb dvc

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.1/470.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.3/79.3 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 451.2/451.2 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.7/45.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.8/155.8 kB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 214.2/214.2 kB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.3/57.3 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.2/74.2 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 381.2/38

**Step 2**: downloading two full years (2023 and 2024) of yellow-taxi data. Each file is compressed at this step and only about ~50MB each.

In [3]:
import os, urllib.request
os.makedirs("data", exist_ok=True)
base = "https://d37ci6vzurychx.cloudfront.net/trip-data"
for year in (2023, 2024):
  for m in range(1, 13):
    fn=f"yellow_tripdata_{year}-{m:02d}.parquet"
    dest=f"data/{fn}"
    if not os.path.exists(dest):
      print("Downloading...", fn)
      urllib.request.urlretrieve(f"{base}/{fn}", dest)
print("Done")

Downloading... yellow_tripdata_2023-01.parquet
Downloading... yellow_tripdata_2023-02.parquet
Downloading... yellow_tripdata_2023-03.parquet
Downloading... yellow_tripdata_2023-04.parquet
Downloading... yellow_tripdata_2023-05.parquet
Downloading... yellow_tripdata_2023-06.parquet
Downloading... yellow_tripdata_2023-07.parquet
Downloading... yellow_tripdata_2023-08.parquet
Downloading... yellow_tripdata_2023-09.parquet
Downloading... yellow_tripdata_2023-10.parquet
Downloading... yellow_tripdata_2023-11.parquet
Downloading... yellow_tripdata_2023-12.parquet
Downloading... yellow_tripdata_2024-01.parquet
Downloading... yellow_tripdata_2024-02.parquet
Downloading... yellow_tripdata_2024-03.parquet
Downloading... yellow_tripdata_2024-04.parquet
Downloading... yellow_tripdata_2024-05.parquet
Downloading... yellow_tripdata_2024-06.parquet
Downloading... yellow_tripdata_2024-07.parquet
Downloading... yellow_tripdata_2024-08.parquet
Downloading... yellow_tripdata_2024-09.parquet
Downloading..

How much disk space used and how many files:

In [4]:
!du -sh data
!ls data | wc -l

1.3G	data
24


**Step 3:** measuring one month of data and extrapolating the memory requirement of all 24. The problem is proved in this step, as we see there is more data to be analyzed than there is memory.

In [5]:
import pandas as pd
df = pd.read_parquet("data/yellow_tripdata_2024-01.parquet")
gb = df.memory_usage(deep=True).sum() / 1e9
print(f"{df.shape[0]:,} rows, {gb:.2f} GB in RAM for ONE month")
print(f"{gb*24:.1f} GB if all 24 months are loaded - vs Colab's ~12.7 GB")

2,964,624 rows, 0.54 GB in RAM for ONE month
12.9 GB if all 24 months are loaded - vs Colab's ~12.7 GB


**Step 4**: since Parquet is column-oriented, we can read just the columns we're interested in (vs CSV which is row-based and requires reading every row to get a full column of info). By reading just the columns we're interested in (tpep_pickup_datetime and fare_amount), we significantly reduce the amount of memory we need to use.

In [6]:
small = pd.read_parquet(
    "data/yellow_tripdata_2024-01.parquet",
    columns=["tpep_pickup_datetime", "fare_amount"]
)
print(small.memory_usage(deep=True).sum()/1e6, "MB vs", gb*1000, "MB full")

47.434116 MB vs 535.917488 MB full


**Step 5**: Out-of-core querying of all 24 files at once using DuckDB. DuckDB processes Parquet files in a stream - pulling data in batches, computing aggregation incrementally, and discarding the batches it's done with. The SQL query below aggregates ~100M rows while barely using any RAM. (Notice the RAM gauge in Colab staying flat while this runs.)

In [7]:
import duckdb
con = duckdb.connect()

#total trips + avg fare across ALL 24 months
con.sql("""
  SELECT count(*) AS trips, avg(fare_amount) AS avg_fare
  FROM 'data/yellow_tripdata_*.parquet'
""").show()

┌──────────┬────────────────────┐
│  trips   │      avg_fare      │
│  int64   │       double       │
├──────────┼────────────────────┤
│ 79479946 │ 19.390815694714938 │
└──────────┴────────────────────┘



Analytical query of tipping behavior by hour of day:

In [8]:
con.sql("""
  SELECT date_part('hour', tpep_pickup_datetime) AS hour,
    count(*) AS trips,
    round(avg(tip_amount), 2) AS avg_tip
  FROM 'data/yellow_tripdata_*.parquet'
  GROUP BY hour ORDER BY hour
""").show()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

┌───────┬─────────┬─────────┐
│ hour  │  trips  │ avg_tip │
│ int64 │  int64  │ double  │
├───────┼─────────┼─────────┤
│     0 │ 2280685 │    3.29 │
│     1 │ 1506367 │    2.96 │
│     2 │  987554 │    2.74 │
│     3 │  651432 │    2.79 │
│     4 │  461309 │    3.29 │
│     5 │  490803 │    3.91 │
│     6 │ 1118584 │    3.32 │
│     7 │ 2178121 │    3.14 │
│     8 │ 3008709 │    3.12 │
│     9 │ 3341800 │     3.2 │
│     · │    ·    │      ·  │
│     · │    ·    │      ·  │
│     · │    ·    │      ·  │
│    14 │ 4744123 │    3.58 │
│    15 │ 4883423 │    3.56 │
│    16 │ 4935930 │    3.79 │
│    17 │ 5395258 │    3.59 │
│    18 │ 5659952 │    3.42 │
│    19 │ 5014943 │    3.42 │
│    20 │ 4511897 │    3.44 │
│    21 │ 4556966 │    3.47 │
│    22 │ 4222771 │    3.48 │
│    23 │ 3313700 │     3.5 │
├───────┴─────────┴─────────┤
│    24 rows (20 shown)     │
└───────────────────────────┘



**Step 6:** partition pruning - organizing data so queries only read what they need. In this case, repartitioning the data by year and month.

In [9]:
con.sql("""
  COPY(
    SELECT *,
      date_part('year', tpep_pickup_datetime) AS year,
      date_part('month', tpep_pickup_datetime) as month,
    FROM  'data/yellow_tripdata_*.parquet'
  ) TO 'data_partitioned'
  (FORMAT PARQUET, PARTITION_BY (year, month), OVERWRITE_OR_IGNORE)
""")
!find data_partitioned -maxdepth 2 | head

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

data_partitioned
data_partitioned/year=2022
data_partitioned/year=2022/month=10
data_partitioned/year=2022/month=12
data_partitioned/year=2014
data_partitioned/year=2014/month=11
data_partitioned/year=2026
data_partitioned/year=2026/month=6
data_partitioned/year=2008
data_partitioned/year=2008/month=12


Now, a filtered query doesn't touch all 24 months, but only a relevant folder:

In [10]:
con.sql("""
  SELECT count(*) FROM 'data_partitioned/*/*/*.parquet'
  WHERE year=2024 and month=7
""").show()

┌──────────────┐
│ count_star() │
│    int64     │
├──────────────┤
│      3076876 │
└──────────────┘



**Step 7:** Versioning the data using DVC which works by replacing the folder with a text pointer that Git tracks while the actual bytes stay in a cache.

In [11]:
!git init -q
!dvc init -q
!dvc add data_partitioned. # creates data_partitioned.dvc (a small pointer)
!cat data_partitioned.dvc # see the pointer: hash + size, not the data

⠋ Checking graph
Adding...:   0% 0/1 [00:00<?, ?file/s{'info': ''}]
!
          |0.00 [00:00,     ?file/s]
100% 6.00/6.00 [00:00<00:00, 44.3file/s{'info': ''}]
100% 11.0/11.0 [00:00<00:00, 13.2file/s{'info': ''}]
100% 14.0/14.0 [00:01<00:00, 10.6file/s{'info': ''}]
100% 16.0/16.0 [00:01<00:00, 10.0file/s{'info': ''}]
100% 18.0/18.0 [00:01<00:00, 11.4file/s{'info': ''}]
100% 20.0/20.0 [00:01<00:00, 11.0file/s{'info': ''}]
100% 22.0/22.0 [00:01<00:00, 10.5file/s{'info': ''}]
100% 24.0/24.0 [00:02<00:00, 10.0file/s{'info': ''}]
100% 26.0/26.0 [00:02<00:00, 9.42file/s{'info': ''}]
100% 28.0/28.0 [00:02<00:00, 9.32file/s{'info': ''}]
100% 29.0/29.0 [00:02<00:00, 9.23file/s{'info': ''}]
100% 30.0/30.0 [00:02<00:00, 9.32file/s{'info': ''}]
                                                    
!
  0% |          |0/? [00:00<?,    ?files/s]
                                           
Adding data_partitioned to cache:   0% 0/36 [00:00<?, ?file/s]
Adding data_partitioned to cache:   0% 0/36 [00:00<